In [0]:
# ==========================================================
# ENVIRONMENT & PIPELINE CONTEXT INITIALIZATION
# ==========================================================

# Purpose:
# - Fetch environment & pipeline metadata from upstream task
# - Set them as environment variables for downstream usage
# - Ensure safe defaults and debugging visibility

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

import os, sys

# ==========================================================
# FETCH VALUES FROM INIT MASTER TASK
# ==========================================================
env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

# ==========================================================
# SET ENV VARIABLES (GLOBAL CONTEXT)
# ==========================================================

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")

from superstore_logger import get_superstore_logger, log_event
# -----------------------------
# log_event Setup for catalog and schema init
# -----------------------------
# Initialize log_event to capture events in the gold dimensions pipeline
logger_catalog_schema = get_superstore_logger("superstore_catalog_and_schemas_init")

# --------------------------------------------------------------
# Log Context
# --------------------------------------------------------------
log_event(
    logger_catalog_schema,
    "INFO",
    "Starting catalog and schema initialization",
    layer="catalog_and_schema_init",
    env=env,
    pipeline=pipeline_name,
    pipeline_version=pipeline_version
)

# ==========================================================
# 1. DEFINE CATALOG & SCHEMAS
# ==========================================================

catalog_name = "superstore_catalog"

data_domains = [
    "audit",
    "bronze",
    "silver",
    "gold",
    "features",
    "mart",
    "metrics",
    "quarantine",
    "semantic_layer"
]

schemas = [f"{env}_{domain}" for domain in data_domains]

log_event(
    logger_catalog_schema,
    "INFO",
    "Prepared catalog and schema list",
    layer="catalog_and_schema_init",
    catalog=catalog_name,
    schemas=",".join(schemas),
    env=env
)

# ==========================================================
# 2. CREATE CATALOG
# ==========================================================
#
# Create only when missing. `CREATE CATALOG IF NOT EXISTS` is still a CREATE
# statement: running it every time means the identity running the pipeline must
# hold CREATE CATALOG on the metastore forever, for something needed once per
# environment. A service principal should not hold that -- and on this
# workspace it cannot: the metastore is Databricks-owned (Free Edition), so the
# privilege is not grantable. Checking first means a pipeline identity that can
# merely SEE the catalog runs cleanly, while a missing catalog is still created
# by an identity allowed to, exactly as before.
#
# SHOW CATALOGS lists only catalogs the caller can see, so "not listed" means
# missing OR invisible; the CREATE that follows fails loudly in the second case.

try:
    catalog_exists = spark.sql(f"SHOW CATALOGS LIKE '{catalog_name}'").count() > 0
    if not catalog_exists:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

    log_event(
        logger_catalog_schema,
        "INFO",
        "Catalog ensured successfully",
        layer="catalog_and_schema_init",
        catalog=catalog_name,
        env=env
    )

except Exception as e:
    log_event(
        logger_catalog_schema,
        "ERROR",
        "Failed to create catalog",
        layer="catalog_and_schema_init",
        catalog=catalog_name,
        error=str(e),
        env=env
    )
    raise


# ==========================================================
# 3. CREATE SCHEMAS
# ==========================================================

for domain in data_domains:
    schema_name = f"{env}_{domain}"
    full_schema = f"{catalog_name}.{schema_name}"

    # Same reasoning as the catalog: CREATE_SCHEMA is needed only when the
    # schema is actually missing, which for prod is never after its first run.
    try:
        schema_existed = spark.catalog.databaseExists(full_schema)
        if not schema_existed:
            spark.sql(f"CREATE SCHEMA IF NOT EXISTS {full_schema}")

        log_event(
            logger_catalog_schema,
            "INFO",
            "Schema ensured successfully",
            layer="catalog_and_schema_init",
            schema=full_schema,
            created=not schema_existed,
            env=env
        )

    except Exception as e:
        log_event(
            logger_catalog_schema,
            "ERROR",
            "Failed to create schema",
            
            schema=full_schema,
            error=str(e),
            env=env
        )
        raise


# ==========================================================
# 4. VALIDATION
# ==========================================================

try:
    schemas_df = spark.sql(f"SHOW SCHEMAS IN {catalog_name}")
    schema_count = schemas_df.count()

    log_event(
        logger_catalog_schema,
        "INFO",
        "Schema validation completed",
        layer="catalog_and_schema_init",
        catalog=catalog_name,
        total_schemas=schema_count,
        env=env
    )

except Exception as e:
    log_event(
        logger_catalog_schema,
        "ERROR",
        "Schema validation failed",
        layer="catalog_and_schema_init",
        catalog=catalog_name,
        error=str(e),
        env=env
    )
    raise


# --------------------------------------------------------------
# Completion Log
# --------------------------------------------------------------
log_event(
    logger_catalog_schema,
    "INFO",
    "Catalog and schema initialization completed successfully",
    layer="catalog_and_schema_init",
    catalog=catalog_name,
    env=env
)